In [74]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import re

In [75]:
raw = pd.read_csv(
    "../data/data1.1.csv",
    encoding="utf-8-sig",
    header=None,
    dtype=str,
    keep_default_na=False,
)
print("raw shape:", raw.shape)

raw shape: (1245, 1757)


In [76]:
raw_region = pd.read_csv(
    "../data/data1.1.csv",
    encoding="utf-8-sig",
    header=None,
    usecols=[0],
    dtype=str,
    skip_blank_lines=False,
).iloc[:, 0]

raw_full = pd.read_csv(
    "../data/data1.1.csv",
    encoding="utf-8-sig",
    header=None,
    usecols=[0, 1],
    dtype=str,
    skip_blank_lines=False,
)

print("Rows where col0 is non-null but col1 is null/blank (likely markers):")
markers = raw_full[raw_full[1].isna() | (raw_full[1].astype(str).str.strip() == "")]
print(markers.to_string())

Rows where col0 is non-null but col1 is null/blank (likely markers):
                             0    1
1                          NaN  NaN
2                  SDG REGIONS  NaN
211                        NaN  NaN
212   OTHER REGIONAL GROUPINGS  NaN
317                        NaN  NaN
318           INCOME GROUPINGS  NaN
449                        NaN  NaN
450   UNICEF PROGRAMME REGIONS  NaN
633                        NaN  NaN
634   UNICEF REPORTING REGIONS  NaN
921                        NaN  NaN
922                WHO REGIONS  NaN
1079                       NaN  NaN
1106                       NaN  NaN
1108                       NaN  NaN
1109                       NaN  NaN
1110                       NaN  NaN
1111                       NaN  NaN
1112                       NaN  NaN
1113                       NaN  NaN
1114                       NaN  NaN
1115                       NaN  NaN
1116                       NaN  NaN
1117                       NaN  NaN
1118                       NaN 

In [77]:
region_vals = raw_full[0].dropna().astype(str).str.strip()
region_vals = region_vals[region_vals != ""]


import re
def looks_like_marker(s):
    if re.search(r"(SDG|REGION|TOTAL|WORLD|INCOME|GROUP|SUBTOTAL|^[A-Z\s&\-]{4,}$)", s):
        return True
    if s.endswith(":"):
        return True
    return False

candidates = [v for v in region_vals.unique() if looks_like_marker(v)]
print("Marker candidates:")
for c in candidates:
    print("  ", repr(c))

Marker candidates:
   'REGION'
   'SDG REGIONS'
   'OTHER REGIONAL GROUPINGS'
   'INCOME GROUPINGS'
   'UNICEF PROGRAMME REGIONS'
   'UNICEF REPORTING REGIONS'
   'WHO REGIONS'
   'WORLD'
   'Key:'


In [78]:
SECTION_MARKERS = [
    "SDG REGIONS",
    "OTHER REGIONAL GROUPINGS",
    "INCOME GROUPINGS",
    "UNICEF PROGRAMME REGIONS",
    "UNICEF REPORTING REGIONS",
    "WHO REGIONS",
    "WORLD",
]
NON_DATA_FOOTERS = {"Key:"}   

all_markers = set(SECTION_MARKERS) | NON_DATA_FOOTERS

print("Markers to treat as section headers:", SECTION_MARKERS)
print("Footer rows to drop:", NON_DATA_FOOTERS)

Markers to treat as section headers: ['SDG REGIONS', 'OTHER REGIONAL GROUPINGS', 'INCOME GROUPINGS', 'UNICEF PROGRAMME REGIONS', 'UNICEF REPORTING REGIONS', 'WHO REGIONS', 'WORLD']
Footer rows to drop: {'Key:'}


In [79]:
top = raw.iloc[0].copy()
sub = raw.iloc[1].copy()

keep = ~((top.str.strip() == "") & (sub.str.strip() == ""))   # boolean Series aligned to columns

top  = top[keep]
sub  = sub[keep]
body = raw.iloc[2:].loc[:, keep].reset_index(drop=True)


top_ff = top.str.strip().replace("", np.nan).ffill()
simple = ["REGION", "Year", "School age-population (thousands)",
          "% urban", "% pre-primary", "% primary", "% secondary"]
group_start = list(top_ff).index("TOTAL")

cols = []
for i in range(len(top_ff)):
    g = str(top_ff.iloc[i]).strip()
    s = str(sub.iloc[i]).strip()
    if i < group_start:
        cols.append(simple[i])
    else:
        if s == "":
            cols.append(g)
        else:
            if   "Basic"   in s: suffix = "basic"
            elif "Limited" in s: suffix = "limited"
            elif "No water" in s or "No facility" in s: suffix = "none"
            else: suffix = re.sub(r"\W+", "_", s).strip("_").lower()
            cols.append(f"{g}__{suffix}")

body.columns = cols
print("body shape:", body.shape)
print(body.columns.tolist()[:20])

body shape: (1243, 25)
['REGION', 'Year', 'School age-population (thousands)', '% urban', '% pre-primary', '% primary', '% secondary', 'TOTAL__basic', 'TOTAL__limited', 'TOTAL__none', 'URBAN__basic', 'URBAN__limited', 'URBAN__none', 'RURAL__basic', 'RURAL__limited', 'RURAL__none', 'PRE-PRIMARY__basic', 'PRE-PRIMARY__limited', 'PRE-PRIMARY__none', 'PRIMARY__basic']


In [80]:
body["REGION"] = body["REGION"].str.strip()
body["Year"]   = body["Year"].str.strip()

current_group = None
groups   = []
is_data  = []
for _, row in body.iterrows():
    r = row["REGION"]
    y = row["Year"]
    if r in SECTION_MARKERS:
        current_group = r
        groups.append(None)
        is_data.append(False)
    elif r in NON_DATA_FOOTERS:
        groups.append(None)
        is_data.append(False)
    elif r == "REGION" and y == "Year":       
        groups.append(None)
        is_data.append(False)
    else:
        groups.append(current_group)
        year_ok = pd.to_numeric(y, errors="coerce")
        is_data.append(pd.notna(year_ok) and 1900 <= year_ok <= 2100)

body["region_grouping"] = groups
body["_is_data"]        = is_data

print("Data rows   :", body["_is_data"].sum())
print("Non-data rows:", (~body["_is_data"]).sum())
print("\nRows per grouping:")
print(body[body["_is_data"]].groupby("region_grouping").size())

Data rows   : 1066
Non-data rows: 177

Rows per grouping:
region_grouping
INCOME GROUPINGS            130
OTHER REGIONAL GROUPINGS    104
SDG REGIONS                 208
UNICEF PROGRAMME REGIONS    182
UNICEF REPORTING REGIONS    286
WHO REGIONS                 156
dtype: int64


In [81]:
for g, sub_df in body[body["_is_data"]].groupby("region_grouping", sort=False):
    uniq = sorted(sub_df["REGION"].unique())
    print(f"\n=== {g} ({len(uniq)} regions) ===")
    for r in uniq:
        print("  ", r)


=== SDG REGIONS (8 regions) ===
   Australia and New Zealand
   Central and Southern Asia
   Eastern and South-Eastern Asia
   Europe and Northern America
   Latin America and the Caribbean
   Northern Africa and Western Asia
   Oceania
   Sub-Saharan Africa

=== OTHER REGIONAL GROUPINGS (4 regions) ===
   Highly and extremely fragile
   Landlocked developing countries
   Least developed countries
   Small island developing States

=== INCOME GROUPINGS (5 regions) ===
   High-income
   Low- and middle-income
   Low-income
   Lower-middle-income
   Upper-middle-income

=== UNICEF PROGRAMME REGIONS (7 regions) ===
   East Asia and Pacific
   Eastern and Southern Africa
   Europe and Central Asia
   Latin America and the Caribbean
   Middle East and North Africa
   South Asia
   West and Central Africa

=== UNICEF REPORTING REGIONS (11 regions) ===
   East Asia and Pacific
   Eastern Europe and Central Asia
   Eastern and Southern Africa
   Europe and Central Asia
   Latin America and Ca

In [82]:
data = body[body["_is_data"]].drop(columns=["_is_data"]).copy()
data["Year"] = pd.to_numeric(data["Year"], errors="coerce").astype("Int64")

# ---- Clean numeric cells ----
def clean_num_col(s):
    s = (s.astype(str)
           .str.replace("\u00a0", " ", regex=False)
           .str.replace("\u2009", " ", regex=False)
           .str.replace(r"\s+", "", regex=True)
           .str.replace(",", "", regex=False))
    numeric_part = pd.to_numeric(s.str.lstrip("<>"), errors="coerce")
    # keep "<1"/">99" strings; otherwise use numeric
    out = s.where(s.str.match(r"^[<>]"), numeric_part)
    return out

for c in data.columns:
    if c in ("REGION", "Year", "region_grouping"):
        continue
    data[c] = clean_num_col(data[c])

# Convert fully-numeric columns to Float64
for c in data.columns:
    if c in ("REGION", "Year", "region_grouping"):
        continue
    coerced = pd.to_numeric(data[c], errors="coerce")
    if coerced.notna().sum() == data[c].notna().sum():
        data[c] = coerced.astype("Float64")

# Column order: grouping, region, year, meta, then grouped sections
meta_cols   = ["School age-population (thousands)", "% urban",
               "% pre-primary", "% primary", "% secondary"]
group_order = ["TOTAL", "URBAN", "RURAL", "PRE-PRIMARY", "PRIMARY", "SECONDARY"]
def sort_key(c):
    if c == "region_grouping": return (0, 0, c)
    if c == "REGION":          return (0, 1, c)
    if c == "Year":            return (0, 2, c)
    if c in meta_cols:         return (0, 3, meta_cols.index(c), c)
    for g in group_order:
        if c.startswith(g + "__"): return (1, group_order.index(g), c)
    return (2, 0, c)
data = data[sorted(data.columns, key=sort_key)]

print("Shape:", data.shape)
print(data[["region_grouping", "REGION", "Year"]].head(10))
print("\nDtypes:")
print(data.dtypes.value_counts())

Shape: (1066, 26)
   region_grouping                     REGION  Year
1      SDG REGIONS  Australia and New Zealand  2000
2      SDG REGIONS  Australia and New Zealand  2001
3      SDG REGIONS  Australia and New Zealand  2002
4      SDG REGIONS  Australia and New Zealand  2003
5      SDG REGIONS  Australia and New Zealand  2004
6      SDG REGIONS  Australia and New Zealand  2005
7      SDG REGIONS  Australia and New Zealand  2006
8      SDG REGIONS  Australia and New Zealand  2007
9      SDG REGIONS  Australia and New Zealand  2008
10     SDG REGIONS  Australia and New Zealand  2009

Dtypes:
object     14
Float64    11
Int64       1
Name: count, dtype: int64


In [83]:
data.to_csv("../data/data_clean.csv", index=False)
print("Saved ../data/data_clean.csv")

PermissionError: [Errno 13] Permission denied: '../data/data_clean.csv'

# Connecting to Database

In [84]:
!pip install sqlalchemy psycopg2-binary

In [ ]:
from sqlalchemy import create_engine, text
from sqlalchemy.engine import URL
import pandas as pd

url = URL.create(
    drivername="postgresql+psycopg2",
    username="postgres",
    password="safin",
    host="localhost",
    port=5432,
    database="education_data",
)
engine = create_engine(url, pool_pre_ping=True)

# Sanity check
with engine.connect() as conn:
    print(conn.execute(text("SELECT version();")).scalar())

PostgreSQL 18.4 on x86_64-windows, compiled by msvc-19.44.35227, 64-bit


In [ ]:
import re

def to_snake(c):
    c = c.replace("%", "pct_").replace("__", "_")
    c = re.sub(r"[^\w]+", "_", c).strip("_").lower()
    return c

db_df = data.rename(columns=to_snake)

print(db_df.columns.tolist()[:20])

['region_grouping', 'region', 'year', 'school_age_population_thousands', 'pct__urban', 'pct__pre_primary', 'pct__primary', 'pct__secondary', 'total_basic', 'total_limited', 'total_none', 'urban_basic', 'urban_limited', 'urban_none', 'rural_basic', 'rural_limited', 'rural_none', 'pre_primary_basic', 'pre_primary_limited', 'pre_primary_none']


In [ ]:
TABLE  = "education_water_access"
SCHEMA = "public"

db_df.to_sql(
    name=TABLE,
    con=engine,
    schema=SCHEMA,
    if_exists="replace",   # "replace" | "append" | "fail"
    index=False,
    method="multi",        # batched inserts (much faster)
    chunksize=1000,
)
print(f"Wrote {len(db_df):,} rows to {SCHEMA}.{TABLE}")

Wrote 1,066 rows to public.education_water_access


In [ ]:
with engine.connect() as conn:
    count = conn.execute(
        text(f"SELECT COUNT(*) FROM {SCHEMA}.{TABLE};")
    ).scalar()
    print("Rows in DB:", count)

    preview = pd.read_sql(
        text(f'SELECT region_grouping, region, year, total_basic, urban_basic '
             f'FROM {SCHEMA}.{TABLE} LIMIT 10;'),
        conn,
    )
print(preview)

Rows in DB: 1066
  region_grouping                     region  year total_basic urban_basic
0     SDG REGIONS  Australia and New Zealand  2000         >99        None
1     SDG REGIONS  Australia and New Zealand  2001         >99        None
2     SDG REGIONS  Australia and New Zealand  2002         >99        None
3     SDG REGIONS  Australia and New Zealand  2003         >99        None
4     SDG REGIONS  Australia and New Zealand  2004         >99        None
5     SDG REGIONS  Australia and New Zealand  2005         >99        None
6     SDG REGIONS  Australia and New Zealand  2006         >99        None
7     SDG REGIONS  Australia and New Zealand  2007         >99        None
8     SDG REGIONS  Australia and New Zealand  2008         >99        None
9     SDG REGIONS  Australia and New Zealand  2009         >99        None


In [ ]:
TABLE  = "education_water_access"
SCHEMA = "public"

db_df.to_sql(
    name=TABLE,
    con=engine,
    schema=SCHEMA,
    if_exists="replace",   # "replace" | "append" | "fail"
    index=False,
    method="multi",        # batched inserts (much faster)
    chunksize=1000,
)
print(f"Wrote {len(db_df):,} rows to {SCHEMA}.{TABLE}")

Wrote 1,066 rows to public.education_water_access


In [ ]:
with engine.connect() as conn:
    count = conn.execute(
        text(f"SELECT COUNT(*) FROM {SCHEMA}.{TABLE};")
    ).scalar()
    print("Rows in DB:", count)

    preview = pd.read_sql(
        text(f'SELECT region_grouping, region, year, total_basic, urban_basic '
             f'FROM {SCHEMA}.{TABLE} LIMIT 10;'),
        conn,
    )
print(preview)

Rows in DB: 1066
  region_grouping                     region  year total_basic urban_basic
0     SDG REGIONS  Australia and New Zealand  2000         >99        None
1     SDG REGIONS  Australia and New Zealand  2001         >99        None
2     SDG REGIONS  Australia and New Zealand  2002         >99        None
3     SDG REGIONS  Australia and New Zealand  2003         >99        None
4     SDG REGIONS  Australia and New Zealand  2004         >99        None
5     SDG REGIONS  Australia and New Zealand  2005         >99        None
6     SDG REGIONS  Australia and New Zealand  2006         >99        None
7     SDG REGIONS  Australia and New Zealand  2007         >99        None
8     SDG REGIONS  Australia and New Zealand  2008         >99        None
9     SDG REGIONS  Australia and New Zealand  2009         >99        None


In [91]:
with engine.begin() as conn:
    conn.execute(text(f'''
        CREATE INDEX IF NOT EXISTS idx_{TABLE}_grp_year
        ON {SCHEMA}.{TABLE} (region_grouping, year);
    '''))
print("Index created.")

Index created.
